## <font color="#F5AD27">__Avance 2 - Modelo Relacional__</font>


---



#### <font color="#F5AD27">__Importación de librerías__</font>
---

In [27]:
import pandas as pd
import numpy as np
import sqlite3
from graphviz import Digraph
import os

In [28]:
df = pd.read_csv('ventasTransformed.csv')

In [29]:
# Observamos información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   VentaID                  30000 non-null  int64 
 1   FechaVenta               30000 non-null  object
 2   HoraVenta                30000 non-null  object
 3   SucursalNombre           30000 non-null  object
 4   CiudadSucursal           30000 non-null  object
 5   VendedorNombre           30000 non-null  object
 6   ClienteNombre            30000 non-null  object
 7   GeneroCliente            30000 non-null  object
 8   EdadCliente              30000 non-null  int64 
 9   EmailCliente             28437 non-null  object
 10  TelefonoCliente          29705 non-null  object
 11  DireccionCliente         30000 non-null  object
 12  MetodoPago               30000 non-null  object
 13  NombreProducto1          30000 non-null  object
 14  MarcaProducto1           30000 non-nul

In [30]:
df.head()

,VentaID,FechaVenta,HoraVenta,SucursalNombre,CiudadSucursal,VendedorNombre,ClienteNombre,GeneroCliente,EdadCliente,EmailCliente,...,CantidadProducto2,PrecioUnitarioProducto2,SubtotalProducto2,NombreProducto3,MarcaProducto3,CantidadProducto3,PrecioUnitarioProducto3,SubtotalProducto3,DescuentoVenta,TotalVenta
0,1,31/12/2015,5:42:43,TechCore Pereira,Pereira,Amílcar Ortega-Alberto,Bienvenida Nebot-Fiol,F,37,bienvenida19@hotmail.com,...,2,80000000,160000000,Dell Latitude 7420,Dell,1,56000000,56000000,0,312000000
1,10,19/07/2023,14:25:30,TechCore Bogotá #2,Bogotá,Armida Azorin Plaza,Sergio Paulino Cánovas Redondo,M,24,sergio78@hotmail.com,...,2,68000000,136000000,NaN,NaN,0,0,0,0,280000000
2,100,08/04/2015,19:25:27,TechCore Bogotá #1,Bogotá,Evaristo Guillen Peña,Rafa Sanmiguel Calvo,M,18,rafa23@hotmail.com,...,1,30000000,30000000,NaN,NaN,0,0,0,0,54000000
3,1000,15/02/2017,2:00:46,TechCore Medellín #1,Medellín,Ibán López Córdoba,Ciro Valera Pareja,M,18,ciro10@gmail.com,...,2,80000000,160000000,NaN,NaN,0,0,0,0,186000000
4,10000,13/05/2018,16:44:37,TechCore Bogotá #1,Bogotá,María Belén Alegria Camps,Lina Guzmán Lamas,F,20,lina85@hotmail.com,...,2,30000000,60000000,HP Spectre x360,HP,2,52000000,104000000,0,208000000


In [31]:
# Cantidad de registros y atributos de la selección
print(f'🔍La cantidad de registros del dataset es de: {df.shape[0]}.')
print(f'🔍La cantidad de atributos del dataset es de: {df.shape[1]}.')

🔍La cantidad de registros del dataset es de: 30000.
🔍La cantidad de atributos del dataset es de: 30.


In [32]:
# Identificando si hay valores duplicados
duplicados = df.duplicated().any()

if duplicados:
    print("Hay registros duplicados en el DataFrame.")
else:
    print("No hay registros duplicados en el DataFrame.")

No hay registros duplicados en el DataFrame.


In [33]:
print(df.columns)

Index(['VentaID', 'FechaVenta', 'HoraVenta', 'SucursalNombre',
       'CiudadSucursal', 'VendedorNombre', 'ClienteNombre', 'GeneroCliente',
       'EdadCliente', 'EmailCliente', 'TelefonoCliente', 'DireccionCliente',
       'MetodoPago', 'NombreProducto1', 'MarcaProducto1', 'CantidadProducto1',
       'PrecioUnitarioProducto1', 'SubtotalProducto1', 'NombreProducto2',
       'MarcaProducto2', 'CantidadProducto2', 'PrecioUnitarioProducto2',
       'SubtotalProducto2', 'NombreProducto3', 'MarcaProducto3',
       'CantidadProducto3', 'PrecioUnitarioProducto3', 'SubtotalProducto3',
       'DescuentoVenta', 'TotalVenta'],
      dtype='object')


#### <font color="#F5AD27">__Conexión a SQLite__</font>
---

In [34]:
# Se crea la conexión a la base SQLite
conn = sqlite3.connect('modeloVentas.db')
cursor = conn.cursor()

# Limpiar tablas si ya existen (opcional para re-ejecutar sin errores)
tablas = ['DetalleFacturas', 'Facturas', 'Productos', 'Clientes', 'Sucursales', 'Vendedores', 'Ciudades']
for t in tablas:
    cursor.execute(f"DROP TABLE IF EXISTS {t}")

conn.commit()

#### <font color="#F5AD27">__Creación de las tablas con claves primarias y foráneas__</font>
---

In [35]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Ciudades (
    CiudadID INTEGER PRIMARY KEY AUTOINCREMENT,
    NombreCiudad TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Sucursales (
    SucursalID INTEGER PRIMARY KEY AUTOINCREMENT,
    NombreSucursal TEXT UNIQUE,
    CiudadID INTEGER,
    FOREIGN KEY (CiudadID) REFERENCES Ciudades(CiudadID)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Clientes (
    ClienteID INTEGER PRIMARY KEY AUTOINCREMENT,
    NombreCliente TEXT,
    Genero TEXT,
    Edad INTEGER,
    Email TEXT,
    Telefono TEXT,
    Direccion TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Vendedores (
    VendedorID INTEGER PRIMARY KEY AUTOINCREMENT,
    NombreVendedor TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Productos (
    ProductoID INTEGER PRIMARY KEY AUTOINCREMENT,
    NombreProducto TEXT,
    Marca TEXT,
    PrecioUnitario REAL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Facturas (
    FacturaID INTEGER PRIMARY KEY,
    FechaVenta TEXT,
    HoraVenta TEXT,
    Descuento REAL,
    TotalVenta REAL,
    MetodoPago TEXT,
    SucursalID INTEGER,
    ClienteID INTEGER,
    VendedorID INTEGER,
    FOREIGN KEY (SucursalID) REFERENCES Sucursales(SucursalID),
    FOREIGN KEY (ClienteID) REFERENCES Clientes(ClienteID),
    FOREIGN KEY (VendedorID) REFERENCES Vendedores(VendedorID)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS DetalleFacturas (
    DetalleID INTEGER PRIMARY KEY AUTOINCREMENT,
    FacturaID INTEGER,
    ProductoID INTEGER,
    Cantidad REAL,
    Subtotal REAL,
    FOREIGN KEY (FacturaID) REFERENCES Facturas(FacturaID),
    FOREIGN KEY (ProductoID) REFERENCES Productos(ProductoID)
)
""")

conn.commit()
print("✅ Tablas creadas correctamente con AUTOINCREMENT.")


✅ Tablas creadas correctamente con AUTOINCREMENT.


#### <font color="#F5AD27">__Poblar tablas creadas__</font>
---

In [36]:
# Insertar Ciudades únicas
ciudades = df["CiudadSucursal"].unique()
for ciudad in ciudades:
    cursor.execute("INSERT OR IGNORE INTO Ciudades (NombreCiudad) VALUES (?)", (ciudad,))
conn.commit()


In [37]:
# Insertar Sucursales
for _, row in df[["SucursalNombre", "CiudadSucursal"]].drop_duplicates().iterrows():
    cursor.execute("SELECT CiudadID FROM Ciudades WHERE NombreCiudad = ?", (row["CiudadSucursal"],))
    ciudad_id = cursor.fetchone()[0]
    cursor.execute(
        "INSERT OR IGNORE INTO Sucursales (NombreSucursal, CiudadID) VALUES (?, ?)",
        (row["SucursalNombre"], ciudad_id)
    )
conn.commit()

In [38]:
# Insertar Clientes
clientes_cols = ["ClienteNombre", "GeneroCliente", "EdadCliente", "EmailCliente", "TelefonoCliente", "DireccionCliente"]

for _, row in df[clientes_cols].drop_duplicates().iterrows():
    cursor.execute("""
        INSERT OR IGNORE INTO Clientes (NombreCliente, Genero, Edad, Email, Telefono, Direccion)
        VALUES (?, ?, ?, ?, ?, ?)
    """, tuple(row))
conn.commit()

In [39]:
# Insertar Vendedores 
vendedores = df["VendedorNombre"].dropna().unique()
for vendedor in vendedores:
    cursor.execute("INSERT OR IGNORE INTO Vendedores (NombreVendedor) VALUES (?)", (vendedor,))
conn.commit()


In [40]:
# Insertar Productos
productos = []
for i in range(1, 4):
    nombre_col = f"NombreProducto{i}"
    marca_col = f"MarcaProducto{i}"
    precio_col = f"PrecioUnitarioProducto{i}"

    sub_df = df[[nombre_col, marca_col, precio_col]].drop_duplicates()
    sub_df.columns = ["NombreProducto", "Marca", "PrecioUnitario"]
    productos.append(sub_df)

productos_df = pd.concat(productos).drop_duplicates(subset=["NombreProducto", "Marca"]).dropna(subset=["NombreProducto"])

for _, row in productos_df.iterrows():
    cursor.execute("""
        INSERT OR IGNORE INTO Productos (NombreProducto, Marca, PrecioUnitario)
        VALUES (?, ?, ?)
    """, (row["NombreProducto"], row["Marca"], row["PrecioUnitario"]))
conn.commit()


In [41]:
# Insertar Factura y almacenar IDs
factura_ids = []

for _, row in df.iterrows():
    cursor.execute("SELECT SucursalID FROM Sucursales WHERE NombreSucursal = ?", (row["SucursalNombre"],))
    sucursal_id = cursor.fetchone()[0]

    cursor.execute("SELECT ClienteID FROM Clientes WHERE NombreCliente = ?", (row["ClienteNombre"],))
    cliente_id = cursor.fetchone()[0]

    cursor.execute("SELECT VendedorID FROM Vendedores WHERE NombreVendedor = ?", (row["VendedorNombre"],))
    vendedor_id = cursor.fetchone()[0]

    cursor.execute("""
        INSERT INTO Facturas (FechaVenta, HoraVenta, Descuento, TotalVenta, MetodoPago, SucursalID, ClienteID, VendedorID)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (row["FechaVenta"], row["HoraVenta"], row["DescuentoVenta"], row["TotalVenta"],
          row["MetodoPago"], sucursal_id, cliente_id, vendedor_id))
    
    
    factura_ids.append(cursor.lastrowid)

conn.commit()

In [42]:
# Insertar DetalleFacturas
for idx, row in df.iterrows():
    factura_id = factura_ids[idx]

    for i in [1, 2, 3]:
        nombre_col = f"NombreProducto{i}"
        cantidad_col = f"CantidadProducto{i}"
        subtotal_col = f"SubtotalProducto{i}"

        if pd.notna(row[nombre_col]):
            cursor.execute("SELECT ProductoID FROM Productos WHERE NombreProducto = ?", (row[nombre_col],))
            producto_id = cursor.fetchone()[0]

            cursor.execute("""
                INSERT INTO DetalleFacturas (FacturaID, ProductoID, Cantidad, Subtotal)
                VALUES (?, ?, ?, ?)
            """, (factura_id, producto_id, row[cantidad_col], row[subtotal_col]))
conn.commit()

#### <font color="#F5AD27">__Verificaciones de consistencia y control de calidad de los datos__</font>
---

In [43]:
queries = {
    "Facturas sin sucursal": """
        SELECT COUNT(*) FROM Facturas f
        LEFT JOIN Sucursales s ON f.SucursalID = s.SucursalID
        WHERE s.SucursalID IS NULL
    """,
    "Facturas sin cliente": """
        SELECT COUNT(*) FROM Facturas f
        LEFT JOIN Clientes c ON f.ClienteID = c.ClienteID
        WHERE c.ClienteID IS NULL
    """,
    "Facturas sin vendedor": """
        SELECT COUNT(*) FROM Facturas f
        LEFT JOIN Vendedores v ON f.VendedorID = v.VendedorID
        WHERE v.VendedorID IS NULL
    """,
    "Detalles sin factura": """
        SELECT COUNT(*) FROM DetalleFacturas d
        LEFT JOIN Facturas f ON d.FacturaID = f.FacturaID
        WHERE f.FacturaID IS NULL
    """,
    "Detalles sin producto": """
        SELECT COUNT(*) FROM DetalleFacturas d
        LEFT JOIN Productos p ON d.ProductoID = p.ProductoID
        WHERE p.ProductoID IS NULL
    """,
    "Facturas con monto negativo": """
        SELECT COUNT(*) FROM Facturas
        WHERE TotalVenta < 0
    """,
    "Facturas con descuento negativo": """
        SELECT COUNT(*) FROM Facturas
        WHERE Descuento < 0
    """,
    "Detalles con cantidad negativa": """
        SELECT COUNT(*) FROM DetalleFacturas
        WHERE Cantidad < 0
    """,
    "Detalles con subtotal negativo": """
        SELECT COUNT(*) FROM DetalleFacturas
        WHERE Subtotal < 0
    """,
    "Productos sin precio": """
        SELECT COUNT(*) FROM Productos
        WHERE PrecioUnitario IS NULL OR PrecioUnitario <= 0
    """,
    "Clientes sin email": """
        SELECT COUNT(*) FROM Clientes
        WHERE Email IS NULL OR Email = ''
    """,
    "Sucursales sin ciudad": """
        SELECT COUNT(*) FROM Sucursales s
        LEFT JOIN Ciudades c ON s.CiudadID = c.CiudadID
        WHERE c.CiudadID IS NULL
    """
}

print("Verificación de consistencia de datos:\n")
for desc, q in queries.items():
    try:
        result = pd.read_sql_query(q, conn).iloc[0, 0]
        status = "✅" if result == 0 else "⚠️"
        print(f"{status} {desc}: {result}")
    except Exception as e:
        print(f"❌ {desc}: Error -> {e}")

Verificación de consistencia de datos:

✅ Facturas sin sucursal: 0
✅ Facturas sin cliente: 0
✅ Facturas sin vendedor: 0
✅ Detalles sin factura: 0
✅ Detalles sin producto: 0
✅ Facturas con monto negativo: 0
✅ Facturas con descuento negativo: 0
✅ Detalles con cantidad negativa: 0
✅ Detalles con subtotal negativo: 0
✅ Productos sin precio: 0
⚠️ Clientes sin email: 900
✅ Sucursales sin ciudad: 0


In [44]:
# Verificar cantidad de registros en cada tabla
print("Conteo de registros por tabla:\n")
print("-------------------------------\n")

tablas_verificar = ['Ciudades', 'Sucursales', 'Clientes', 'Vendedores', 'Productos', 'Facturas', 'DetalleFacturas']

for tabla in tablas_verificar:
    count_query = f"SELECT COUNT(*) FROM {tabla}"
    count = pd.read_sql_query(count_query, conn).iloc[0, 0]
    print(f"  {tabla}: {count:,} registros")

print(f"\n   Dataset original: {len(df):,} registros")

Conteo de registros por tabla:

-------------------------------

  Ciudades: 4 registros
  Sucursales: 6 registros
  Clientes: 17,453 registros
  Vendedores: 30 registros
  Productos: 40 registros
  Facturas: 30,000 registros
  DetalleFacturas: 60,059 registros

   Dataset original: 30,000 registros


In [45]:
# Verificación de la integridad de montos
print("Verificación de integridad de montos:\n")
print("-------------------------------------\n")

# Compararación del total de ventas
total_ventas_db = pd.read_sql_query("SELECT SUM(TotalVenta) as total FROM Facturas", conn).iloc[0, 0]
total_ventas_df = df['TotalVenta'].sum()

print(f"  • Total ventas en DB: ${total_ventas_db:,.2f}")
print(f"  • Total ventas en CSV: ${total_ventas_df:,.2f}")
print(f"  • Diferencia: ${abs(total_ventas_db - total_ventas_df):,.2f}")

# Compararación suma de subtotales vs total de facturas
verificacion_montos = pd.read_sql_query("""
    SELECT 
        f.FacturaID,
        f.TotalVenta,
        SUM(d.Subtotal) as SumaDetalles,
        ABS(f.TotalVenta - SUM(d.Subtotal)) as Diferencia
    FROM Facturas f
    LEFT JOIN DetalleFacturas d ON f.FacturaID = d.FacturaID
    GROUP BY f.FacturaID
    HAVING ABS(f.TotalVenta - SUM(d.Subtotal)) > 0.01
""", conn)

print(f"\n  • Facturas con diferencias en totales: {len(verificacion_montos)}")
if len(verificacion_montos) > 0:
    print(f"  • Diferencia promedio: ${verificacion_montos['Diferencia'].mean():,.2f}")

Verificación de integridad de montos:

-------------------------------------

  • Total ventas en DB: $4,564,916,700,000.00
  • Total ventas en CSV: $4,564,916,700,000.00
  • Diferencia: $0.00

  • Facturas con diferencias en totales: 29997
  • Diferencia promedio: $66,458,482.51


In [46]:
# Valores nulos 
print("Verificación de valores nulos en las principales tablas:\n")
print("----------------------------------------------------\n")

campos_criticos = {
    "Facturas": ["FechaVenta", "TotalVenta", "SucursalID", "ClienteID"],
    "DetalleFacturas": ["FacturaID", "ProductoID", "Cantidad", "Subtotal"],
    "Productos": ["NombreProducto", "PrecioUnitario"],
    "Clientes": ["NombreCliente"],
    "Sucursales": ["NombreSucursal", "CiudadID"]
}

hay_nulos = False

for tabla, campos in campos_criticos.items():
    for campo in campos:
        null_query = f"SELECT COUNT(*) FROM {tabla} WHERE {campo} IS NULL"
        nulls = pd.read_sql_query(null_query, conn).iloc[0, 0]
        if nulls > 0:
            print(f"  ⚠️ {tabla}.{campo}: {nulls} valores nulos")
            hay_nulos = True

if hay_nulos:
    print("\n Se encontraron valores nulos en las principales tablas")
else:
    print(" No se encontraron valores nulos en las principales tablas")

Verificación de valores nulos en las principales tablas:

----------------------------------------------------

 No se encontraron valores nulos en las principales tablas


In [47]:
# Estadísticas de ventas
print("Estadísticas de ventas:\n")
print("------------------------\n")

stats = pd.read_sql_query("""
    SELECT 
        COUNT(*) as TotalFacturas,
        SUM(TotalVenta) as VentaTotal,
        AVG(TotalVenta) as VentaPromedio,
        MIN(TotalVenta) as VentaMinima,
        MAX(TotalVenta) as VentaMaxima,
        AVG(Descuento) as DescuentoPromedio
    FROM Facturas
""", conn)

print(f"   Total de facturas: {stats['TotalFacturas'].iloc[0]:,}")
print(f"   Venta total: ${stats['VentaTotal'].iloc[0]:,.2f}")
print(f"   Venta promedio: ${stats['VentaPromedio'].iloc[0]:,.2f}")
print(f"   Venta mínima: ${stats['VentaMinima'].iloc[0]:,.2f}")
print(f"   Venta máxima: ${stats['VentaMaxima'].iloc[0]:,.2f}")
print(f"   Descuento promedio: {stats['DescuentoPromedio'].iloc[0]:.2f}%")


Estadísticas de ventas:

------------------------

   Total de facturas: 30,000
   Venta total: $4,564,916,700,000.00
   Venta promedio: $152,163,890.00
   Venta mínima: $17,000,000.00
   Venta máxima: $488,000,000.00
   Descuento promedio: 1.59%


In [48]:
# Top 10 productos más vendidos
print("\nTop 10 productos más vendidos:\n")
top_productos = pd.read_sql_query("""
    SELECT 
        p.NombreProducto,
        p.Marca,
        SUM(d.Cantidad) as CantidadTotal,
        SUM(d.Subtotal) as VentaTotal
    FROM DetalleFacturas d
    JOIN Productos p ON d.ProductoID = p.ProductoID
    GROUP BY p.ProductoID, p.NombreProducto, p.Marca
    ORDER BY CantidadTotal DESC
    LIMIT 10
""", conn)

for idx, row in top_productos.iterrows():
    print(f"  {idx+1}. {row['NombreProducto']} ({row['Marca']})")
    print(f"     Cantidad: {row['CantidadTotal']:,.0f} | Ventas: ${row['VentaTotal']:,.2f}")


Top 10 productos más vendidos:

  1. HP Spectre x360 (HP)
     Cantidad: 7,726 | Ventas: $215,020,000,000.00
  2. Lenovo Legion 5 Pro (Lenovo)
     Cantidad: 5,861 | Ventas: $230,637,600,000.00
  3. Lenovo ThinkPad X1 Carbon (Lenovo)
     Cantidad: 5,814 | Ventas: $222,523,200,000.00
  4. Lenovo Yoga 7i (Lenovo)
     Cantidad: 5,052 | Ventas: $125,862,000,000.00
  5. HP Omen 16 (HP)
     Cantidad: 5,011 | Ventas: $166,686,000,000.00
  6. Lenovo IdeaPad 5 (Lenovo)
     Cantidad: 4,970 | Ventas: $77,142,800,000.00
  7. Dell XPS 13 (Dell)
     Cantidad: 4,526 | Ventas: $119,097,600,000.00
  8. Dell Latitude 7420 (Dell)
     Cantidad: 4,507 | Ventas: $139,496,000,000.00
  9. HP Pavilion 15 (HP)
     Cantidad: 4,029 | Ventas: $78,708,000,000.00
  10. Dell Alienware m15 (Dell)
     Cantidad: 3,658 | Ventas: $159,512,000,000.00


In [49]:
# Cantidad de productos por marca
print("CANTIDAD DE PRODUCTOS POR MARCA")


productos_por_marca = pd.read_sql_query("""
    SELECT 
        Marca,
        COUNT(*) as CantidadProductos
    FROM Productos
    GROUP BY Marca
    ORDER BY CantidadProductos DESC
""", conn)

for idx, row in productos_por_marca.iterrows():
    print(f"{idx+1}. {row['Marca']}: {row['CantidadProductos']} productos")

print(f"\nTotal de marcas: {len(productos_por_marca)}")
print(f"Total de productos: {productos_por_marca['CantidadProductos'].sum()}")

CANTIDAD DE PRODUCTOS POR MARCA
1. Samsung: 4 productos
2. Razer: 4 productos
3. Microsoft: 4 productos
4. MSI: 4 productos
5. Lenovo: 4 productos
6. HP: 4 productos
7. Dell: 4 productos
8. Asus: 4 productos
9. Apple: 4 productos
10. Acer: 4 productos

Total de marcas: 10
Total de productos: 40


#### <font color="#F5AD27">__Creación del diagrama ER__</font>
---

In [50]:
dot = Digraph(comment='Modelo Relacional de Ventas', format='png')
dot.attr(rankdir='TB', bgcolor='white')
dot.attr('node', shape='plaintext', fontname='Arial')

# Obtenemos las tablas (excluyendo sqlite_sequence)
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence';")
tablas = [t[0] for t in cursor.fetchall()]

for tabla in tablas:
    cursor.execute(f"PRAGMA table_info({tabla});")
    columnas = cursor.fetchall()
    
    html_label = f'''<
    <TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4" BGCOLOR="lightblue">
        <TR><TD BGCOLOR="darkblue"><FONT COLOR="white"><B>{tabla}</B></FONT></TD></TR>'''
    
    for col in columnas:
        col_name = col[1]
        col_type = col[2]
        is_pk = '🔑 ' if col[5] == 1 else ''
        html_label += f'<TR><TD ALIGN="LEFT">{is_pk}{col_name} : {col_type}</TD></TR>'
    
    html_label += '</TABLE>>'
    dot.node(tabla, label=html_label)

# Relaciones (claves foráneas)
for tabla in tablas:
    cursor.execute(f"PRAGMA foreign_key_list({tabla});")
    fks = cursor.fetchall()
    for fk in fks:
        tabla_referenciada = fk[2]
        campo_origen = fk[3]
        campo_destino = fk[4]
        dot.edge(tabla_referenciada, tabla, label=f"{campo_destino}", 
                arrowhead='crow', color='gray40', fontsize='10')

# Renderizamos el diagrama
try:
    output_path = dot.render('diagrama_ER_ventas', cleanup=True)
    if os.path.exists('diagrama_ER_ventas.png'):
        print("✅ Diagrama ER generado exitosamente: diagrama_ER_ventas.png")
    else:
        print("✅ Diagrama ER generado exitosamente")
except Exception as e:
    if os.path.exists('diagrama_ER_ventas.png'):
        print("✅ Diagrama ER generado exitosamente: diagrama_ER_ventas.png")
    else:
        print(f"⚠️ Error al generar el diagrama: {e}")

✅ Diagrama ER generado exitosamente: diagrama_ER_ventas.png


#### <font color="#F5AD27">__Exportar todas las tablas a Excel__</font>
---

In [51]:
with pd.ExcelWriter('modeloVentas.xlsx', engine='openpyxl') as writer:
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence';")
    tablas = [t[0] for t in cursor.fetchall()]
    
    # Exporta cada tabla a una hoja diferente
    for tabla in tablas:
        df_tabla = pd.read_sql_query(f"SELECT * FROM {tabla}", conn)
        df_tabla.to_excel(writer, sheet_name=tabla, index=False)
        print(f"{tabla}: {len(df_tabla)} registros exportados")

print("\n Archivo 'modeloVentas.xlsx' generado exitosamente con todas las tablas")

Ciudades: 4 registros exportados
Sucursales: 6 registros exportados
Clientes: 17453 registros exportados
Vendedores: 30 registros exportados
Productos: 40 registros exportados
Facturas: 30000 registros exportados
DetalleFacturas: 60059 registros exportados

 Archivo 'modeloVentas.xlsx' generado exitosamente con todas las tablas


#### <font color="#F5AD27">__Cerrar conexión__</font>
---

In [52]:
conn.close()